# Notebook 11 — Transcript Quantification: TPM, FPKM, and Raw Counts

**Module:** 09 — Genomics and NGS  
**Notebook:** 11 of 16  
**Time estimate:** 75 minutes

> **Track A critical:** You will be asked "what is TPM and why is it preferred
> over RPKM?" in almost every RNA-seq interview. Know the formulas and the
> reason, not just the names.

---
## Step 1 — Motivation

Raw read counts are not comparable between genes (longer genes get more reads)
or between samples (deeper sequencing gives more counts). Normalization removes
these biases so we can ask: is gene A expressed more than gene B? Is gene A
expressed more in condition 1 than condition 2?

---
## Step 2 — Intuition

**Two biases to correct:**
1. **Gene length:** A 10 kb gene receives ~10× more reads than a 1 kb gene with
   the same expression level (more positions for reads to land)
2. **Sequencing depth:** A sample with 100M reads has ~2× more counts than one
   with 50M reads for the same gene at the same expression level

**RPKM/FPKM (Reads/Fragments Per Kilobase per Million):**
Correct depth first, then length. The problem: after depth correction, the sum
of RPKM values differs between samples (longer transcriptomes get larger totals).
You cannot directly compare RPKM values across samples.

**TPM (Transcripts Per Million):**
Correct length first, then depth. Result: TPM values in every sample sum to exactly
1,000,000. TPM represents the proportion of transcripts that are from gene $g$,
scaled to 1M. Directly comparable across samples.

**The rule:** Use raw counts for differential expression (DESeq2, edgeR). Use TPM
for visualization and cross-sample comparisons.

---
## Step 3 — Biological Background

**featureCounts:** Assigns aligned reads to genomic features (genes/exons) using
a GTF annotation. Handles paired-end reads, strand-specificity, multi-overlapping
reads, and multi-mapping reads.

**Strand-specific RNA-seq protocols:**
- **Unstranded:** Cannot tell which strand the transcript came from
- **Forward-stranded (dUTP):** Read 1 is antisense, read 2 is sense
- **Reverse-stranded (ligation):** Read 1 is sense, read 2 is antisense

Using the wrong strand setting in featureCounts will assign reads to wrong genes
and dramatically reduce assignment rates. Always check with `infer_experiment.py`
(from RSeQC).

**Effective gene length:**  
The effective length accounts for reads that cannot start near the end of a
transcript (read would extend past transcript end). For read length $L$ and
transcript length $T$:
$$L_{\text{eff}} = T - L + 1$$

---
## Step 4 — Mathematical Explanation

Let:
- $c_g$ = raw count for gene $g$
- $L_g$ = gene length (bp)
- $N$ = total mapped reads in sample

**RPKM:**
$$\text{RPKM}_g = \frac{c_g}{(L_g / 1000) \times (N / 10^6)} = \frac{10^9 \cdot c_g}{L_g \cdot N}$$

**FPKM:** Same formula, but counts fragments (pairs) instead of reads.

**TPM — the two-step calculation:**

Step 1 — rate $r_g$ (reads per kilobase, length-normalized):
$$r_g = \frac{c_g}{L_g / 1000}$$

Step 2 — normalize so all rates sum to $10^6$:
$$\text{TPM}_g = \frac{r_g}{\sum_{g'} r_{g'}} \times 10^6$$

**The key difference:** TPM divides by $\sum r_{g'}$ (the sum of length-normalized
counts in that sample). This sum is proportional to the total transcriptome size,
making TPM values comparable across samples with different gene sets.

In [ ]:
# Step 6 — Python: TPM, FPKM, RPKM from scratch
import numpy as np
from dataclasses import dataclass


def compute_rpkm(counts: np.ndarray, lengths_bp: np.ndarray) -> np.ndarray:
    """
    Compute RPKM for each gene.
    counts: raw read counts per gene
    lengths_bp: gene lengths in base pairs
    """
    N = counts.sum()  # total mapped reads
    if N == 0:
        return np.zeros_like(counts, dtype=float)
    per_million = N / 1e6
    per_kilobase = lengths_bp / 1000
    return counts / (per_million * per_kilobase)


def compute_tpm(counts: np.ndarray, lengths_bp: np.ndarray) -> np.ndarray:
    """
    Compute TPM for each gene.
    Step 1: divide by length (get reads per kilobase = rate)
    Step 2: divide by sum of rates, multiply by 1e6
    """
    per_kilobase = lengths_bp / 1000
    rate = counts / per_kilobase
    rate_sum = rate.sum()
    if rate_sum == 0:
        return np.zeros_like(counts, dtype=float)
    return (rate / rate_sum) * 1e6


def compute_fpkm(counts: np.ndarray, lengths_bp: np.ndarray) -> np.ndarray:
    """FPKM = RPKM for paired-end (fragments instead of reads)."""
    return compute_rpkm(counts, lengths_bp)


# Verify: TPM sums to 1,000,000
rng = np.random.default_rng(42)
n_genes = 20000
gene_lengths = rng.integers(500, 10000, n_genes)
true_expression = rng.exponential(2, n_genes)  # true expression levels (relative)
# Simulate counts: proportional to expression × length (longer gene → more reads)
expected_reads = true_expression * gene_lengths / 1000
counts = rng.poisson(expected_reads * 50).astype(float)  # 50× scaling

rpkm = compute_rpkm(counts, gene_lengths)
tpm = compute_tpm(counts, gene_lengths)

print(f"Sum of raw counts: {counts.sum():,.0f}")
print(f"Sum of RPKM:       {rpkm.sum():,.1f}  (NOT 1,000,000)")
print(f"Sum of TPM:        {tpm.sum():,.1f}  (always exactly 1,000,000)")
print()
print("Spearman correlation (true expression vs. normalized):")
from scipy.stats import spearmanr
r_raw, _ = spearmanr(true_expression, counts)
r_rpkm, _ = spearmanr(true_expression, rpkm)
r_tpm, _ = spearmanr(true_expression, tpm)
print(f"  Raw counts vs. true: {r_raw:.4f}")
print(f"  RPKM vs. true:       {r_rpkm:.4f}")
print(f"  TPM vs. true:        {r_tpm:.4f}")

In [ ]:
# Cross-sample comparability: TPM vs RPKM
# Simulate two samples with same true expression but different gene sets
def simulate_two_samples(n_shared: int = 500, n_unique: int = 200, seed: int = 42):
    rng = np.random.default_rng(seed)
    # Shared genes
    shared_lengths = rng.integers(500, 5000, n_shared)
    shared_expr = rng.exponential(3, n_shared)
    shared_counts_s1 = rng.poisson(shared_expr * shared_lengths / 1000 * 30)
    shared_counts_s2 = rng.poisson(shared_expr * shared_lengths / 1000 * 30)

    # Sample 2 has additional highly expressed genes (inflates total)
    extra_lengths = rng.integers(2000, 8000, n_unique)
    extra_expr = rng.exponential(10, n_unique)  # highly expressed
    extra_counts_s2 = rng.poisson(extra_expr * extra_lengths / 1000 * 30)

    # Sample 1: just shared genes
    s1_counts = shared_counts_s1.astype(float)
    s1_lengths = shared_lengths

    # Sample 2: shared + extra
    s2_counts = np.concatenate([shared_counts_s2, extra_counts_s2]).astype(float)
    s2_lengths = np.concatenate([shared_lengths, extra_lengths])

    return (s1_counts, s1_lengths, shared_counts_s1,
            s2_counts, s2_lengths, shared_counts_s2, shared_lengths)


s1_counts, s1_len, s1_shared, s2_counts, s2_len, s2_shared, shared_len = simulate_two_samples()

# Compute RPKM and TPM for shared genes only
s1_rpkm = compute_rpkm(s1_counts, s1_len)
s2_rpkm_all = compute_rpkm(s2_counts, s2_len)
s2_rpkm_shared = s2_rpkm_all[:len(s1_counts)]

s1_tpm = compute_tpm(s1_counts, s1_len)
s2_tpm_all = compute_tpm(s2_counts, s2_len)
s2_tpm_shared = s2_tpm_all[:len(s1_counts)]

from scipy.stats import pearsonr
r_rpkm_cross, _ = pearsonr(np.log1p(s1_rpkm), np.log1p(s2_rpkm_shared))
r_tpm_cross, _ = pearsonr(np.log1p(s1_tpm), np.log1p(s2_tpm_shared))

print("Cross-sample correlation (shared genes, same true expression):")
print(f"  Pearson r (log RPKM): {r_rpkm_cross:.4f}")
print(f"  Pearson r (log TPM):  {r_tpm_cross:.4f}")
print()
print(f"Mean RPKM S1: {s1_rpkm.mean():.2f}, Mean RPKM S2 (shared): {s2_rpkm_shared.mean():.2f}")
print(f"Mean TPM S1:  {s1_tpm.mean():.2f}, Mean TPM S2 (shared): {s2_tpm_shared.mean():.2f}")
print("(TPM means should be more similar — because TPM normalizes out the extra genes in S2)")

In [ ]:
# Step 7 — Visualization
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel A: Raw counts vs gene length (showing the bias)
ax = axes[0, 0]
sample_idx = rng.integers(0, n_genes, 2000)
ax.scatter(gene_lengths[sample_idx], counts[sample_idx],
           alpha=0.3, s=5, color='gray')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Gene length (bp)')
ax.set_ylabel('Raw read count')
r, _ = spearmanr(gene_lengths[sample_idx], counts[sample_idx])
ax.set_title(f'A. Raw counts correlate with gene length\n(Spearman ρ={r:.2f} — bias!)')

# Panel B: TPM vs gene length (bias removed)
ax2 = axes[0, 1]
ax2.scatter(gene_lengths[sample_idx], tpm[sample_idx],
            alpha=0.3, s=5, color='steelblue')
ax2.set_xscale('log'); ax2.set_yscale('log')
ax2.set_xlabel('Gene length (bp)')
ax2.set_ylabel('TPM')
r2, _ = spearmanr(gene_lengths[sample_idx], tpm[sample_idx])
ax2.set_title(f'B. TPM is independent of gene length\n(Spearman ρ={r2:.2f} — bias removed!)')

# Panel C: TPM vs RPKM cross-sample scatter
ax3 = axes[1, 0]
s_idx = rng.integers(0, len(s1_rpkm), 500)
ax3.scatter(np.log1p(s1_rpkm[s_idx]), np.log1p(s2_rpkm_shared[s_idx]),
            alpha=0.4, s=8, color='tomato', label=f'RPKM (r={r_rpkm_cross:.3f})')
ax3.scatter(np.log1p(s1_tpm[s_idx]), np.log1p(s2_tpm_shared[s_idx]),
            alpha=0.4, s=8, color='steelblue', label=f'TPM (r={r_tpm_cross:.3f})')
ax3.set_xlabel('Sample 1 (log)')
ax3.set_ylabel('Sample 2 (log)')
ax3.set_title('C. TPM vs RPKM: cross-sample comparability\n(S2 has extra highly-expressed genes)')
ax3.legend(fontsize=9)

# Panel D: TPM normalization steps illustrated
ax4 = axes[1, 1]
ax4.axis('off')
steps_text = [
    ('Step', 'Formula', 'Sums to'),
    ('1. Raw counts', 'c_g', 'N (total reads)'),
    ('2. Per-kb rate', 'r_g = c_g / (L_g/1000)', 'Varies by sample'),
    ('3. TPM', 'r_g / Σr × 1e6', '1,000,000 exactly'),
    ('', '', ''),
    ('RPKM', 'c_g / (N/1e6) / (L_g/1000)', 'Varies by sample'),
    ('TPM', 'r_g / Σr_g × 1e6', '1,000,000 always'),
]
t = ax4.table(steps_text, loc='center', cellLoc='left')
t.auto_set_font_size(False); t.set_fontsize(9)
t.scale(1, 1.7)
t[0,0].set_facecolor('#D0E8FF'); t[0,1].set_facecolor('#D0E8FF'); t[0,2].set_facecolor('#D0E8FF')
ax4.set_title('D. TPM calculation steps and comparison with RPKM', fontweight='bold', fontsize=10)

plt.suptitle('Transcript Quantification: TPM vs RPKM vs Raw Counts', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('transcript_quantification.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Gene A: 1000 bp, 100 counts. Gene B: 5000 bp, 500 counts. Total 1M reads.
   Compute RPKM and TPM for both. Which gene is more highly expressed per TPM?
2. Explain why you should NOT use TPM as input to DESeq2.
3. Two samples have 50M and 100M reads respectively. Both have gene X at 500 counts.
   Compute RPKM for both. Then compute TPM for both (assume gene X is 2 kb and
   total RPK sum is 5000 for sample 1, 5200 for sample 2). Which is more comparable?

---
## Step 10 — Quiz

1. What two biases does TPM correct for?
2. Why does the sum of RPKM values differ between samples?
3. Why must you use raw counts for DESeq2 input, not TPM?
4. What is effective transcript length and why does it matter?
5. Gene A has TPM=100 in sample 1 and TPM=200 in sample 2. Can you conclude it
   is 2× more expressed in sample 2? What caveats apply?

---
## Step 12 — Reflection

> *[In one paragraph: explain to a colleague why you use raw counts for DESeq2
> but TPM for a scatter plot comparing expression across samples. What property
> of DESeq2's model makes raw counts the right input?]*

---
*Next: `12_differential_expression_deseq2_statistics.ipynb`*